In [80]:
import streamlit as st
import pandas as pd
import time

import requests
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

In [81]:
# Selenium으로 HTML 가져오기
service = Service(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

url = 'https://www.saramin.co.kr/'
driver.get(url)

wait = WebDriverWait(driver, 10)


In [82]:
# 검색창 클릭
search_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="btn_search"]')))
search_button.click()
time.sleep(2)
print("검색창을 클릭했습니다.")


# 데이터분석 입력
search_box = driver.find_element(By.CSS_SELECTOR, '#ipt_keyword_recruit')
search_box.send_keys('데이터분석') 
time.sleep(2)
print('검색어 입력 완료') 

# # 검색 버튼 클릭
search_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//*[@id="btn_search_recruit"]')))
search_button.click()
time.sleep(2)
print("검색 버튼을 클릭했습니다.")

검색창을 클릭했습니다.
검색어 입력 완료
검색 버튼을 클릭했습니다.


In [83]:
# 채용 정보 추출
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
items = soup.find_all('div', class_='item_recruit')

saramin_data = []

for item in items:  
    try:
        # 회사 추출
        corp_element = item.find('strong', class_='corp_name')
        company = corp_element.get_text().strip() if corp_element else ''

        # 제목 추출
        title_element = item.find('h2', class_='job_tit')
        title = title_element.get_text().strip() if title_element else ''

        # detail 추출
        condition_element = item.find('div', class_='job_condition')
        conditions = condition_element.find_all('span')
        detail = []
        for condition in conditions:
            detail.append(condition.get_text().strip())
        
        # url 추출
        link_element = item.find('a', target='_blank')
        url = link_element.get('href') if link_element else ''     


        # 데이터 저장
        saramin_data.append({
            'Site': 'Saramin',
            'Col_Company': company,
            'Col_Recuit': title,
            'Col_detail': detail,
            'Col_url': "https://www.saramin.co.kr" + url
        })
        
    except Exception as e:
        print(f"데이터 추출 중 오류 발생: {e}")
        continue

# 종료
driver.quit()

# 데이터프레임 생성
saramin_df = pd.DataFrame(saramin_data)

# 데이터프레임 출력
saramin_df.head()

,Site,Col_Company,Col_Recuit,Col_detail,Col_url
0,Saramin,NH농협은행(주),NH농협은행 데이터사업부 데이터분석지원 일반계약직 직원 채용,"[서울 중구, 경력무관, 학력무관, 계약직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
1,Saramin,NH농협은행(주),NH농협은행 전문계약직(데이터 분석) 채용공고,"[서울 중구, 경력무관, 학력무관, 계약직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
2,Saramin,에이치에스효성첨단소재(주),2025년 하반기 HS효성그룹 신입공채(IT기술_생산데이터분석),"[서울전체, 신입, 대졸↑, 정규직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
3,Saramin,피엠인터내셔널코리아(유),데이터 분석/데이터 관리/영업/영업 분석/전략 기획/경영 기획,"[서울 영등포구, 신입·경력, 대졸↑, 정규직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...
4,Saramin,(주)심네트,"데이터 분석(국방분야) 전문 분석관 모집(IT, 데이터 분석)","[서울 용산구, 신입·경력, 대졸↑, 정규직·계약직]",https://www.saramin.co.kr/zf_user/jobs/relay/v...


In [84]:
# CSV 파일로 저장
saramin_df.to_csv('data_temp/data_saramin.csv', index=False, encoding='utf-8-sig')
print(f"총 {len(saramin_df)}개의 채용공고가 저장되었습니다.")

총 40개의 채용공고가 저장되었습니다.
